# VISART craft classifier

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**

Keep the tab open. The download is the slow part.

In [ ]:
!nvidia-smi -L || echo 'NO GPU: change the runtime type before training'
!pip install -q open_clip_torch

## Project scripts
These cells only write files. Do not edit them here.

In [ ]:
%%writefile manifest.py
"""Build and check the dataset manifest.

Usage:  python manifest.py [data_folder]      (default: data)

Photos live in <data>/images/<class>/<object_id>/<file>. New photos are added to <data>/manifest.csv;
fill in the source and licence columns there. Exits with status 1 while there are problems.
"""

import csv
import hashlib
import sys
from pathlib import Path

from PIL import Image

CLASSES = ("pottery", "wood", "textile", "metal", "bamboo")
FIELDS = ("file", "class", "object_id", "source", "licence", "author")
EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
MIN_SIDE = 224
TARGET_PER_CLASS = 300


def read_manifest(root: Path) -> list[dict]:
    path = Path(root) / "manifest.csv"
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle))


def write_manifest(root: Path, rows: list[dict]) -> None:
    with (Path(root) / "manifest.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, FIELDS, extrasaction="ignore", lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def check(root: Path) -> dict:
    root = Path(root)
    rows = {row["file"]: row for row in read_manifest(root)}
    on_disk = sorted(
        p.relative_to(root).as_posix()
        for p in (root / "images").rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    problems = []
    hashes: dict[str, str] = {}
    object_classes: dict[str, set] = {}
    counts = {name: {"images": 0, "objects": set()} for name in CLASSES}

    for file in on_disk:
        parts = file.split("/")
        if len(parts) != 4:
            problems.append(f"{file}: expected images/<class>/<object_id>/<file>")
            continue
        _, craft, object_id, _ = parts
        if craft not in CLASSES:
            problems.append(f"{file}: unknown class '{craft}'")
            continue

        row = rows.setdefault(file, {"file": file, "source": "", "licence": "", "author": ""})
        row["class"], row["object_id"] = craft, object_id
        if not (row.get("source") or "").strip() or not (row.get("licence") or "").strip():
            problems.append(f"{file}: no source or licence")

        try:
            with Image.open(root / file) as image:
                image.load()
                if min(image.size) < MIN_SIDE:
                    problems.append(f"{file}: too small (under {MIN_SIDE} px)")
        except (OSError, ValueError):
            problems.append(f"{file}: cannot be read as an image")
            continue

        digest = hashlib.sha256((root / file).read_bytes()).hexdigest()
        if digest in hashes:
            problems.append(f"{file}: same image as {hashes[digest]}")
        hashes.setdefault(digest, file)

        object_classes.setdefault(object_id, set()).add(craft)
        counts[craft]["images"] += 1
        counts[craft]["objects"].add(object_id)

    for object_id, crafts in object_classes.items():
        if len(crafts) > 1:
            problems.append(f"object {object_id} is in more than one class: {', '.join(sorted(crafts))}")
    for file in sorted(set(rows) - set(on_disk)):
        problems.append(f"{file}: file is missing (delete its manifest row if intended)")

    if root.exists():
        write_manifest(root, [rows[file] for file in sorted(rows)])

    return {
        "counts": {name: {"images": c["images"], "objects": len(c["objects"])} for name, c in counts.items()},
        "problems": problems,
    }


def main(argv: list[str]) -> int:
    root = Path(argv[1] if len(argv) > 1 else "data")
    if not (root / "images").is_dir():
        print(f"No photos yet. Put them in {root / 'images'}/<class>/<object_id>/")
        print("Classes: " + ", ".join(CLASSES))
        return 1

    report = check(root)
    total = 0
    print(f"{'class':<10}{'images':>8}{'objects':>9}  target {TARGET_PER_CLASS}")
    for name, count in report["counts"].items():
        total += count["images"]
        print(f"{name:<10}{count['images']:>8}{count['objects']:>9}")
    print(f"{'total':<10}{total:>8}")
    for problem in report["problems"][:50]:
        print("PROBLEM", problem)
    print(f"{len(report['problems'])} problem(s)")
    return 1 if report["problems"] else 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))


In [ ]:
%%writefile collect.py
"""Download openly licensed craft photos with their source, licence and author.

Usage:  python collect.py [data_folder]

Sources: Cleveland Museum of Art (CC0), The Met (CC0), Openverse and Wikimedia Commons (CC0,
public domain, CC BY, CC BY-SA). A source that asks for a long pause is left alone for this run. Safe to run again: files already listed in the manifest are not downloaded twice.
"""

import hashlib
import io
import itertools
import json
import re
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from PIL import Image

from manifest import CLASSES, MIN_SIDE, read_manifest, write_manifest

AGENT = {"User-Agent": "visart-student-dataset/0.1 (educational image classification)"}
PER_CLASS = {"commons": 320, "cleveland": 0, "met": 0, "openverse": 0}  # 0: slow, gone or refusing (see DATA.md)
PREFIX = {"commons": "wc-", "cleveland": "cma-", "met": "met-", "openverse": "ov-"}
PAUSE = {"commons": 1.5, "openverse": 0.7, "cleveland": 0.2, "met": 0.2}
MAX_BYTES = 15 * 1024 * 1024
WORKERS = {"commons": 1, "cleveland": 4, "met": 3, "openverse": 4}  # Wikimedia: one at a time

COMMONS_QUERIES = {
    "pottery": ["terracotta pot", "clay pot handmade", "pottery India", "ceramic vase", "terracotta lamp", "earthenware jug"],
    "wood": ["wooden toy", "wood carving object", "carved wooden box", "wooden spoon handmade", "Channapatna toys", "wooden bowl"],
    "textile": ["saree fabric", "handwoven textile", "embroidered cloth", "woven shawl", "ikat fabric", "block printed fabric"],
    "metal": ["brass lamp", "brass vessel", "copper pot", "bronze figurine", "Bidriware", "brass handicraft"],
    "bamboo": ["bamboo basket", "cane basket", "bamboo handicraft", "wicker basket", "rattan basket", "bamboo craft product"],
}
CLEVELAND_TYPES = {"pottery": ("Ceramic", ""), "wood": ("", "wood"), "textile": ("Textile", ""),
                   "metal": ("Metalwork", ""), "bamboo": ("", "bamboo basket")}
MET_QUERIES = {"pottery": "terracotta vessel", "wood": "wood box", "textile": "textile fragment cotton",
               "metal": "brass vessel", "bamboo": "bamboo basket"}


def allowed_licence(name: str) -> bool:
    return bool(re.fullmatch(r"(?i)(cc0.*|public domain|cc by(-sa)? \d\.\d.*|pd.*)", name.strip()))


class RateLimited(Exception):
    """The site asked us to wait too long; stop using it."""


def object_id_for(uploader: str, title: str, prefix: str = "wc-") -> str:
    """Files in a numbered series from one uploader are treated as the same object."""
    stem = re.sub(r"\.\w+$", "", title.removeprefix("File:"))
    stem = re.sub(r"[\d()\[\]_\-.,#]+", " ", stem).lower().split()
    key = uploader + "|" + " ".join(stem)
    return prefix + hashlib.sha1(key.encode()).hexdigest()[:10]


def commons_record(page: dict) -> dict | None:
    info = (page.get("imageinfo") or [{}])[0]
    meta = info.get("extmetadata", {})
    licence = meta.get("LicenseShortName", {}).get("value", "")
    if info.get("mime") not in ("image/jpeg", "image/png") or not info.get("thumburl"):
        return None
    if min(info.get("width", 0), info.get("height", 0)) < 400 or not allowed_licence(licence):
        return None
    author = re.sub(r"<[^>]+>", "", meta.get("Artist", {}).get("value", "")).strip()
    return {
        "url": info["thumburl"],
        "source": info.get("descriptionurl", ""),
        "licence": licence,
        "author": " ".join((author or info.get("user", "")).split())[:120],
        "object_id": object_id_for(info.get("user", ""), page["title"]),
        "name": f"wc-{page['pageid']}",
    }


def openverse_record(item: dict) -> dict | None:
    kind = item.get("license", "")
    version = item.get("license_version") or ""
    licence = {"cc0": "CC0", "pdm": "Public domain"}.get(kind, f"CC {kind.upper()} {version}")
    if not allowed_licence(licence) or min(item.get("width") or 0, item.get("height") or 0) < 400:
        return None
    author = " ".join((item.get("creator") or item.get("source") or "").split())[:120]
    return {
        "url": item["url"],
        "source": item.get("foreign_landing_url", ""),
        "licence": licence,
        "author": author,
        "object_id": object_id_for(author, item.get("title") or item["id"], "ov-"),
        "name": f"ov-{item['id']}",
    }


def fetch(url: str, tries: int = 4, timeout: int = 40) -> bytes:
    for attempt in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=AGENT), timeout=timeout) as response:
                return response.read(MAX_BYTES + 1)
        except urllib.error.HTTPError as error:
            wait = int(error.headers.get("Retry-After") or 0)
            if error.code == 429 and (wait > 30 or attempt == tries - 1):
                raise RateLimited(f"{urllib.parse.urlsplit(url).netloc} asked to wait {wait}s") from error
            if error.code not in (429, 500, 502, 503) or attempt == tries - 1:
                raise
            time.sleep(wait or 5 * (attempt + 1))
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == tries - 1:
                raise
            time.sleep(3 * (attempt + 1))
    return b""


def get_json(url: str, params: dict) -> dict:
    return json.loads(fetch(url + "?" + urllib.parse.urlencode(params)))


def commons(craft: str):
    per_query = PER_CLASS["commons"] // len(COMMONS_QUERIES[craft]) + 15
    for query in COMMONS_QUERIES[craft]:
        offset, yielded = 0, 0
        while yielded < per_query and offset < 400:
            data = get_json("https://commons.wikimedia.org/w/api.php", {
                "action": "query", "format": "json", "generator": "search", "gsrnamespace": 6,
                "gsrsearch": f"{query} filetype:bitmap", "gsrlimit": 50, "gsroffset": offset,
                "prop": "imageinfo", "iiprop": "url|mime|size|user|extmetadata", "iiurlwidth": 500,  # a standard Wikimedia thumbnail size
            })
            pages = sorted(data.get("query", {}).get("pages", {}).values(), key=lambda p: p.get("index", 0))
            for page in pages:
                record = commons_record(page)
                if record and yielded < per_query:
                    yielded += 1
                    yield record
            offset = data.get("continue", {}).get("gsroffset")
            if offset is None:
                break


def cleveland(craft: str):
    kind, query = CLEVELAND_TYPES[craft]
    params = {"cc0": 1, "has_image": 1, "limit": PER_CLASS["cleveland"] + 20}
    if kind:
        params["type"] = kind
    if query:
        params["q"] = query
    for item in get_json("https://openaccess-api.clevelandart.org/api/artworks/", params).get("data", []):
        url = ((item.get("images") or {}).get("web") or {}).get("url")
        if url:
            yield {"url": url, "source": item.get("url", ""), "licence": "CC0", "author": "Cleveland Museum of Art",
                   "object_id": f"cma-{item['id']}", "name": f"cma-{item['id']}"}


def met(craft: str):
    base = "https://collectionapi.metmuseum.org/public/collection/v1"
    ids = get_json(f"{base}/search", {"hasImages": "true", "q": MET_QUERIES[craft]}).get("objectIDs") or []
    for object_id in ids[: PER_CLASS["met"] * 2]:
        try:
            item = json.loads(fetch(f"{base}/objects/{object_id}"))
        except (urllib.error.URLError, ValueError):
            continue
        time.sleep(0.1)
        if item.get("isPublicDomain") and item.get("primaryImageSmall"):
            yield {"url": item["primaryImageSmall"], "source": item.get("objectURL", ""), "licence": "CC0",
                   "author": "The Metropolitan Museum of Art", "object_id": f"met-{object_id}", "name": f"met-{object_id}"}


def openverse(craft: str):
    for query in COMMONS_QUERIES[craft]:
        for page in range(1, 6):
            data = get_json("https://api.openverse.org/v1/images/", {
                "q": query, "license": "cc0,pdm,by,by-sa", "category": "photograph", "mature": "false",
                "excluded_source": "wikimedia", "page_size": 20, "page": page,
            })
            time.sleep(1)
            for item in data.get("results", []):
                record = openverse_record(item)
                if record:
                    yield record
            if page >= data.get("page_count", 0):
                break


def save(root: Path, craft: str, record: dict) -> dict | None:
    data = fetch(record["url"], tries=2, timeout=15)  # skip slow files; there are plenty
    if len(data) > MAX_BYTES:
        return None
    with Image.open(io.BytesIO(data)) as image:
        image = image.convert("RGB")
        image.thumbnail((640, 640))
        if min(image.size) < MIN_SIDE:
            return None
        relative = f"images/{craft}/{record['object_id']}/{record['name']}.jpg"
        (root / relative).parent.mkdir(parents=True, exist_ok=True)
        image.save(root / relative, "JPEG", quality=90)  # re-encoding drops any metadata or payload
    return {"file": relative, "class": craft, "object_id": record["object_id"],
            "source": record["source"], "licence": record["licence"], "author": record["author"]}


def main(argv: list[str]) -> int:
    root = Path(argv[1] if len(argv) > 1 else "data")
    root.mkdir(parents=True, exist_ok=True)
    rows = {row["file"]: row for row in read_manifest(root)}
    seen = {row["source"] for row in rows.values()}
    blocked = set()

    for craft in CLASSES:
        for source in (s for s in (cleveland, met, openverse, commons) if PER_CLASS[s.__name__]):
            if source.__name__ in blocked:
                continue
            limit = PER_CLASS[source.__name__]
            prefix = "/" + PREFIX[source.__name__]
            got = sum(1 for r in rows.values() if r["class"] == craft and prefix in r["file"].rsplit(r["object_id"], 1)[-1])
            failed = 0

            def attempt(record):
                try:
                    return record, save(root, craft, record), None
                except (OSError, ValueError, Image.DecompressionBombError) as error:  # includes URLError
                    return record, None, error
                finally:
                    time.sleep(PAUSE[source.__name__])

            try:
                records = (r for r in source(craft) if r["source"] not in seen)
                with ThreadPoolExecutor(WORKERS[source.__name__]) as pool:
                    while got < limit:
                        batch = list(itertools.islice(records, min(WORKERS[source.__name__] * 3, limit - got)))
                        if not batch:
                            break
                        for record, row, error in pool.map(attempt, batch):
                            if error:
                                failed += 1
                                continue
                            seen.add(record["source"])
                            if row:
                                rows[row["file"]] = row
                                got += 1
                        write_manifest(root, [rows[key] for key in sorted(rows)])
                        print(f"  {craft:<8} {source.__name__:<9} {got:>4}/{limit}", flush=True)
            except RateLimited as error:
                blocked.add(source.__name__)
                print(f"{craft}/{source.__name__}: {error}; skipping this source", flush=True)
            except (urllib.error.URLError, ValueError, TimeoutError, ConnectionError) as error:
                print(f"{craft}/{source.__name__}: stopped early: {error}", flush=True)
            write_manifest(root, [rows[key] for key in sorted(rows)])
            print(f"{craft:<8} {source.__name__:<9} {got:>4} images ({failed} failed)", flush=True)
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))


In [ ]:
%%writefile embedder.py
"""CLIP image embeddings (ViT-B/32, 512 numbers). Pretrained; nothing is trained here."""

import open_clip
import torch
from PIL import Image

MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"
VERSION = f"clip-{MODEL_NAME}-{PRETRAINED}"


class Embedder:
    def __init__(self):
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device).eval()

    @torch.inference_mode()
    def embed(self, images: list[Image.Image]) -> torch.Tensor:
        """Unit-length vectors, one row per image."""
        batch = torch.stack([self.preprocess(image.convert("RGB")) for image in images])
        return torch.nn.functional.normalize(self.model.encode_image(batch.to(self.device)), dim=1).cpu()

    @torch.inference_mode()
    def embed_text(self, prompts: list[str]) -> torch.Tensor:
        tokens = open_clip.get_tokenizer(MODEL_NAME)(prompts)
        return torch.nn.functional.normalize(self.model.encode_text(tokens.to(self.device)), dim=1).cpu()


In [ ]:
%%writefile clean.py
"""Move downloaded photos that are clearly not a craft object into <data>/rejected/.

Usage:  python clean.py [data_folder]

Only downloaded files are checked (never our own photos). It rejects obvious non-objects, such as
people or buildings. It does not check whether a photo has the right class.
"""

import sys
from pathlib import Path

from PIL import Image

from embedder import Embedder
from manifest import read_manifest, write_manifest

WANTED = [
    "a photo of a pottery or ceramic object", "a photo of a wooden object or wood carving",
    "a photo of a textile, fabric or saree", "a photo of a metal object such as brass or copper",
    "a photo of a bamboo, cane or wicker object",
]
JUNK = [
    "a photo of a person or a group of people", "a photo of a building or a room", "a landscape or street scene",
    "a scanned document, text page, map or diagram", "a painting, drawing or print on paper",
    "a photo of an animal", "a photo of living plants or a forest", "a photo of food", "a photo of a vehicle",
]
REJECT_ABOVE = 0.6
DOWNLOADED = ("wc-", "cma-", "met-", "ov-")


def main(argv):
    root = Path(argv[1] if len(argv) > 1 else "data")
    rows = read_manifest(root)
    targets = [r for r in rows if r["object_id"].startswith(DOWNLOADED) and (root / r["file"]).is_file()]
    embedder = Embedder()
    text = embedder.embed_text(WANTED + JUNK)

    rejected = set()
    for start in range(0, len(targets), 64):
        chunk = targets[start:start + 64]
        images = [Image.open(root / r["file"]).convert("RGB") for r in chunk]
        junk = (100 * embedder.embed(images) @ text.T).softmax(1)[:, len(WANTED):].sum(1)
        for row, score in zip(chunk, junk.tolist()):
            if score > REJECT_ABOVE:
                rejected.add(row["file"])
        print(f"checked {start + len(chunk)}/{len(targets)}", flush=True)

    for file in rejected:
        target = root / "rejected" / file.removeprefix("images/")
        target.parent.mkdir(parents=True, exist_ok=True)
        (root / file).replace(target)
    write_manifest(root, [r for r in rows if r["file"] not in rejected])
    print(f"rejected {len(rejected)} of {len(targets)} downloaded photos")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))


In [ ]:
%%writefile split.py
"""Split the dataset into train, validation and test by object.

Usage:  python split.py [data_folder]

Writes <data>/split.csv. Files already listed there never move, so the test set stays frozen.
"""

import csv
import random
import sys
from pathlib import Path

from manifest import read_manifest

SHARES = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 2026


def assign_splits(rows: list[dict], previous: dict[str, str]) -> dict[str, str]:
    result: dict[str, str] = {}
    for craft in sorted({r["class"] for r in rows}):
        objects: dict[str, list[str]] = {}
        for row in rows:
            if row["class"] == craft:
                objects.setdefault(row["object_id"], []).append(row["file"])

        counts = dict.fromkeys(SHARES, 0)
        new = []
        for object_id in sorted(objects):
            files = objects[object_id]
            old = next((previous[f] for f in sorted(files) if f in previous), None)
            if old:
                counts[old] += len(files)
                result.update(dict.fromkeys(files, old))
            else:
                new.append(files)

        random.Random(f"{SEED}-{craft}").shuffle(new)
        new.sort(key=len, reverse=True)  # place large groups while there is room (stable sort)
        total = sum(len(f) for f in objects.values())
        for files in new:
            # Give the object to whichever split is furthest below its share.
            name = max(SHARES, key=lambda s: SHARES[s] * total - counts[s])
            counts[name] += len(files)
            result.update(dict.fromkeys(files, name))
    return result


def read_split(root: Path) -> dict[str, str]:
    path = Path(root) / "split.csv"
    if not path.exists():
        return {}
    with path.open(newline="", encoding="utf-8") as handle:
        return {row["file"]: row["split"] for row in csv.DictReader(handle)}


def main(argv: list[str]) -> int:
    root = Path(argv[1] if len(argv) > 1 else "data")
    rows = [r for r in read_manifest(root) if (root / r["file"]).is_file()]
    if not rows:
        print("The manifest is empty. Run manifest.py or collect.py first.")
        return 1
    result = assign_splits(rows, read_split(root))
    with (root / "split.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(("file", "split"))
        writer.writerows(sorted(result.items()))
    for name in SHARES:
        print(f"{name:<6}{sum(1 for v in result.values() if v == name):>6}")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))


In [ ]:
%%writefile classifier.py
"""Craft type classifier: model building, saving, loading, prediction and scoring."""

import numpy as np
import torch
from PIL import Image
from torchvision import models
from torchvision.transforms import v2

from manifest import CLASSES

ARCHITECTURES = {
    "mobilenet_v3_large": (models.mobilenet_v3_large, models.MobileNet_V3_Large_Weights.IMAGENET1K_V2),
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
}
NORMALIZE = v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
PREPARE = v2.Compose([
    v2.ToImage(), v2.Resize(256, antialias=True), v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True), NORMALIZE,
])


def build(arch: str, pretrained: bool = True) -> torch.nn.Module:
    factory, weights = ARCHITECTURES[arch]
    model = factory(weights=weights if pretrained else None)
    last = model.classifier[-1]
    model.classifier[-1] = torch.nn.Linear(last.in_features, len(CLASSES))
    return model


def save(path, model, arch: str, threshold: float, version: str) -> None:
    torch.save({"state": model.state_dict(), "arch": arch, "classes": list(CLASSES),
                "threshold": float(threshold), "version": version}, path)


class Classifier:
    def __init__(self, model, classes, threshold, version):
        self.model, self.classes, self.threshold, self.version = model.eval(), classes, threshold, version

    @classmethod
    def load(cls, path):
        data = torch.load(path, map_location="cpu", weights_only=True)  # never unpickle arbitrary code
        model = build(data["arch"], pretrained=False)
        model.load_state_dict(data["state"])
        return cls(model, data["classes"], data["threshold"], data["version"])

    @torch.inference_mode()
    def predict(self, image: Image.Image) -> dict:
        probabilities = self.model(PREPARE(image.convert("RGB")).unsqueeze(0)).softmax(1)[0]
        confidence, index = probabilities.max(0)
        label = self.classes[int(index)] if float(confidence) >= self.threshold else "uncertain"
        return {"label": label, "confidence": round(float(confidence), 4)}


def pick_threshold(confidence: np.ndarray, correct: np.ndarray, target: float = 0.9, floor: float = 0.5) -> float:
    """Lowest confidence at which accepted predictions reach the target precision."""
    for value in np.unique(np.concatenate([[floor], confidence[confidence >= floor]])):
        accepted = confidence >= value
        if accepted.any() and correct[accepted].mean() >= target:
            return round(float(value), 4)
    return 0.95


def scores(confusion, classes) -> dict:
    """Rows are true classes, columns are predictions."""
    matrix = np.asarray(confusion, dtype=float)
    per_class, f1_values = {}, []
    for i, name in enumerate(classes):
        hit = matrix[i, i]
        precision = hit / matrix[:, i].sum() if matrix[:, i].sum() else 0.0
        recall = hit / matrix[i].sum() if matrix[i].sum() else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1_values.append(f1)
        per_class[name] = {"precision": round(float(precision), 3), "recall": round(float(recall), 3),
                           "f1": round(float(f1), 3), "support": int(matrix[i].sum())}
    return {"per_class": per_class, "macro_f1": round(float(np.mean(f1_values)), 3),
            "accuracy": round(float(np.trace(matrix) / matrix.sum()), 3) if matrix.sum() else 0.0}


In [ ]:
%%writefile train.py
"""Fine-tune the craft classifier, compare architectures and write a report.

Usage:  python train.py [data_folder]

Saves models/classifier.pt (best validation macro-F1) and reports/classifier.md. The test split is
used once, for the chosen model only.
"""

import json
import os
import random
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torchvision.transforms import v2

from classifier import ARCHITECTURES, NORMALIZE, PREPARE, build, pick_threshold, save, scores
from manifest import CLASSES, read_manifest
from split import read_split

SEED = 2026
BATCH = 32
HEAD_EPOCHS = int(os.environ.get("HEAD_EPOCHS", 3))
TUNE_EPOCHS = int(os.environ.get("TUNE_EPOCHS", 10))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATIENCE = 3

AUGMENT = v2.Compose([
    v2.RandomResizedCrop(224, scale=(0.6, 1.0), antialias=True), v2.RandomHorizontalFlip(),
    v2.RandomRotation(10), v2.ColorJitter(0.3, 0.3, 0.2, 0.03),
    v2.RandomGrayscale(0.05), v2.ToDtype(torch.float32, scale=True), NORMALIZE,
])


def load(root: Path, split_name: str):
    """Returns (resized uint8 images for augmentation, ready tensors, labels)."""
    split = read_split(root)
    raw, ready, labels = [], [], []
    for row in read_manifest(root):
        if split.get(row["file"]) != split_name:
            continue
        with Image.open(root / row["file"]) as image:
            image = image.convert("RGB")
            ready.append(PREPARE(image))
            raw.append(v2.functional.to_image(image.resize((256, 256))))
        labels.append(CLASSES.index(row["class"]))
    return raw, torch.stack(ready), torch.tensor(labels)


@torch.inference_mode()
def predict(model, images):
    model.eval()
    return torch.cat([model(images[i:i + 64].to(DEVICE)).softmax(1).cpu() for i in range(0, len(images), 64)])


def confusion(truth, predicted):
    matrix = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    for t, p in zip(truth.tolist(), predicted.tolist()):
        matrix[t, p] += 1
    return matrix


def run_epoch(model, optimizer, loss_fn, raw, labels, frozen):
    model.train()
    if frozen:
        model.features.eval()  # keep batch-norm statistics while only the head learns
    order = torch.randperm(len(labels))
    total = 0.0
    for start in range(0, len(order), BATCH):
        index = order[start:start + BATCH]
        if len(index) < 2:
            continue
        batch = torch.stack([AUGMENT(raw[i]) for i in index.tolist()]).to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(batch), labels[index].to(DEVICE))
        loss.backward()
        optimizer.step()
        total += float(loss) * len(index)
    return total / len(labels)


def train_one(arch, train_set, val_set):
    torch.manual_seed(SEED)
    random.seed(SEED)
    raw, _, labels = train_set
    _, val_images, val_labels = val_set
    model = build(arch).to(DEVICE)
    counts = torch.bincount(labels, minlength=len(CLASSES)).float()
    loss_fn = torch.nn.CrossEntropyLoss(weight=(counts.sum() / (len(CLASSES) * counts.clamp(min=1))).to(DEVICE),
                                        label_smoothing=0.05)
    best, best_state, waited = -1.0, None, 0

    for stage, epochs in (("head", HEAD_EPOCHS), ("fine-tune", TUNE_EPOCHS)):
        for parameter in model.features.parameters():
            parameter.requires_grad = False
        if stage == "fine-tune":
            for block in list(model.features)[-4:]:
                for parameter in block.parameters():
                    parameter.requires_grad = True
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=1e-3 if stage == "head" else 2e-4, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
        waited = 0
        for epoch in range(epochs):
            started = time.time()
            loss = run_epoch(model, optimizer, loss_fn, raw, labels, frozen=stage == "head")
            scheduler.step()
            predicted = predict(model, val_images).argmax(1)
            f1 = scores(confusion(val_labels, predicted), CLASSES)["macro_f1"]
            print(f"{arch} {stage} {epoch + 1}/{epochs} loss {loss:.3f} val macro-F1 {f1:.3f} "
                  f"({time.time() - started:.0f}s)", flush=True)
            if f1 > best:
                best, waited = f1, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                waited += 1
                if stage == "fine-tune" and waited >= PATIENCE:
                    break
    model.load_state_dict(best_state)
    return model, best


def report(path, results, arch, threshold, test, matrix, covered, accepted_precision, sizes):
    lines = [f"# Craft classifier, {date.today().isoformat()}", "",
             "**Data:** openly licensed web and museum photos (Wikimedia Commons, Cleveland Museum of Art,",
             "The Met). Labels come from search terms and collection types and were not all hand-checked.",
             "**These numbers are not accuracy on artisan phone photos**, which has not been measured.", "",
             f"Images: train {sizes[0]}, validation {sizes[1]}, test {sizes[2]}.", "",
             "## Model comparison (validation)", "", "| Architecture | Macro-F1 |", "|---|---|"]
    lines += [f"| {name} | {f1:.3f} |" for name, f1 in results.items()]
    lines += ["", f"Chosen: **{arch}**.", "", "## Test set (chosen model, evaluated once)", "",
              f"Macro-F1 **{test['macro_f1']:.3f}**, accuracy {test['accuracy']:.3f}.", "",
              "| Class | Precision | Recall | F1 | Images |", "|---|---|---|---|---|"]
    lines += [f"| {c} | {v['precision']} | {v['recall']} | {v['f1']} | {v['support']} |"
              for c, v in test["per_class"].items()]
    lines += ["", "Confusion matrix (rows are true classes):", "",
              "| | " + " | ".join(CLASSES) + " |", "|---" * (len(CLASSES) + 1) + "|"]
    lines += [f"| **{c}** | " + " | ".join(str(int(v)) for v in matrix[i]) + " |" for i, c in enumerate(CLASSES)]
    lines += ["", "## Uncertain threshold", "",
              f"Chosen on validation: {threshold}. On test, {covered:.0%} of photos get an answer, and "
              f"{accepted_precision:.0%} of those answers are right. The rest are `uncertain`.",
              "Photos of things outside the five classes have not been tested."]
    path.parent.mkdir(exist_ok=True)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def main(argv):
    root = Path(argv[1] if len(argv) > 1 else "data")
    torch.set_num_threads(10)
    sets = {name: load(root, name) for name in ("train", "val", "test")}
    sizes = [len(sets[n][2]) for n in ("train", "val", "test")]
    print("device:", DEVICE, "| images:", dict(zip(sets, sizes)), flush=True)

    trained = {arch: train_one(arch, sets["train"], sets["val"]) for arch in ARCHITECTURES}
    results = {arch: f1 for arch, (_, f1) in trained.items()}
    arch = max(results, key=results.get)
    model = trained[arch][0]

    val_probabilities = predict(model, sets["val"][1])
    confidence, guess = val_probabilities.max(1)
    threshold = pick_threshold(confidence.numpy(), (guess == sets["val"][2]).numpy())

    test_confidence, test_guess = predict(model, sets["test"][1]).max(1)
    matrix = confusion(sets["test"][2], test_guess)
    test = scores(matrix, CLASSES)
    accepted = test_confidence >= threshold
    correct = test_guess == sets["test"][2]
    covered = float(accepted.float().mean())
    precision = float(correct[accepted].float().mean()) if accepted.any() else 0.0

    Path("models").mkdir(exist_ok=True)
    version = f"{arch}-{date.today().isoformat()}"
    save("models/classifier.pt", model, arch, threshold, version)
    report(Path("reports/classifier.md"), results, arch, threshold, test, matrix, covered, precision, sizes)
    print(json.dumps({"version": version, "val": results, "test_macro_f1": test["macro_f1"],
                      "threshold": threshold, "coverage": round(covered, 3),
                      "accepted_precision": round(precision, 3)}), flush=True)
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))


## 1. Download openly licensed photos
Safe to run again: it continues where it stopped.

In [ ]:
!python collect.py data

## 2. Remove obvious junk, then count

In [ ]:
!python clean.py data
!python manifest.py data

## 3. Look at the photos
Delete wrong ones before trusting the results.

In [ ]:
import random, matplotlib.pyplot as plt
from PIL import Image
from manifest import CLASSES, read_manifest
rows = read_manifest('data')
for craft in CLASSES:
    files = [r['file'] for r in rows if r['class'] == craft]
    picks = random.Random(1).sample(files, min(16, len(files)))
    plt.figure(figsize=(16, 4)); plt.suptitle(f'{craft}: {len(files)} images')
    for i, f in enumerate(picks):
        plt.subplot(2, 8, i + 1); plt.imshow(Image.open('data/' + f)); plt.axis('off')
    plt.show()

## 4. Split by object and train

In [ ]:
!python split.py data
!python train.py data

In [ ]:
!cat reports/classifier.md

## 5. Download the results

In [ ]:
!zip -q -r visart_results.zip models reports data/manifest.csv data/split.csv
from google.colab import files
files.download('visart_results.zip')